In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix

# Завантаження даних
ratings = pd.read_csv("bda_dataset5_2_rating.csv")
books = pd.read_csv("bda_dataset5_2_book.csv")

print(ratings.head())
print(books.head())


<h1> Базова інформація

<h1>

In [ ]:
print(f"Унікальних користувачів: {ratings['User-ID'].nunique()}")
print(f"Унікальних книг: {ratings['ISBN'].nunique()}")


Створення матриці користувач-книга

In [ ]:
def create_matrix(df):
    N = df['User-ID'].nunique()
    M = df['ISBN'].nunique()

    user_mapper = dict(zip(np.unique(df["User-ID"]), list(range(N))))
    book_mapper = dict(zip(np.unique(df["ISBN"]), list(range(M))))

    user_inv_mapper = dict(zip(list(range(N)), np.unique(df["User-ID"])))
    book_inv_mapper = dict(zip(list(range(M)), np.unique(df["ISBN"])))

    user_index = [user_mapper[i] for i in df['User-ID']]
    book_index = [book_mapper[i] for i in df['ISBN']]

    X = csr_matrix((df["Book-Rating"], (book_index, user_index)), shape=(M, N))

    return X, user_mapper, book_mapper, user_inv_mapper, book_inv_mapper

X, user_mapper, book_mapper, user_inv_mapper, book_inv_mapper = create_matrix(ratings)


Рекомендації на основі книги

In [ ]:
def find_similar_books(isbn, X, k=10, metric='cosine', show_distance=False):
    neighbour_ids = []

    book_ind = book_mapper[isbn]
    book_vec = X[book_ind]

    kNN = NearestNeighbors(n_neighbors=k+1, algorithm="brute", metric=metric)
    kNN.fit(X)
    book_vec = book_vec.reshape(1, -1)
    neighbour = kNN.kneighbors(book_vec, return_distance=show_distance)

    for i in range(0, k+1):
        n = neighbour.item(i)
        neighbour_ids.append(book_inv_mapper[n])
    neighbour_ids.pop(0) 
    return neighbour_ids

book_titles = dict(zip(books['ISBN'], books['Book-Title']))

isbn = '0385504209'  
similar_ids = find_similar_books(isbn, X, k=10)

print(f"Оскільки Ви читали \"{book_titles[isbn]}\", Вам можуть сподобатись:")
for i in similar_ids:
    print(book_titles.get(i, "Назва відсутня"))


Рекомендації на основі улюбленої книги користувача

In [ ]:
def recommend_books_for_user(user_id, X, user_mapper, book_mapper, book_inv_mapper, k=10):
    df1 = ratings[ratings['User-ID'] == user_id]

    if df1.empty:
        print(f"Користувача з ID {user_id} не знайдено.")
        return

    best_rated = df1[df1['Book-Rating'] == df1['Book-Rating'].max()]
    isbn = best_rated.iloc[0]['ISBN']

    similar_ids = find_similar_books(isbn, X, k)
    title = book_titles.get(isbn, "Книга не знайдена")

    print(f"Оскільки Ви поставили найвищий рейтинг книзі \"{title}\", можливо Вам сподобаються:")
    for i in similar_ids:
        print(book_titles.get(i, "Назва відсутня"))

# Заміни на будь-який реальний ID
recommend_books_for_user(276729, X, user_mapper, book_mapper, book_inv_mapper, k=10)
